# LLM Experiment-5

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 39.7 MB/s eta 0:00:00


In [ ]:
import torch

print("GPU Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

GPU Available: True
GPU: Tesla T4


In [ ]:
import pandas as pd
import random

rows = 4000

data = {
    "transaction_id":[f"T{i}" for i in range(rows)],
    "customer_age":[random.randint(21,65) for _ in range(rows)],
    "income":[random.randint(30000,200000) for _ in range(rows)],
    "credit_score":[random.randint(300,850) for _ in range(rows)],
    "loan_amount":[random.randint(1000,50000) for _ in range(rows)],
    "loan_term":[random.choice([12,24,36,48,60]) for _ in range(rows)],
    "interest_rate":[round(random.uniform(3,15),2) for _ in range(rows)],
    "account_balance":[random.randint(1000,100000) for _ in range(rows)],
    "transaction_type":[random.choice(["deposit","withdrawal","transfer","payment"]) for _ in range(rows)],
}

df = pd.DataFrame(data)

In [ ]:
def assign_risk(row):

    if row["credit_score"] > 700 and row["income"] > 80000:
        return "low"

    elif row["credit_score"] > 600:
        return "medium"

    else:
        return "high"

df["risk_level"] = df.apply(assign_risk, axis=1)

df.head()

,transaction_id,customer_age,income,credit_score,loan_amount,loan_term,interest_rate,account_balance,transaction_type,risk_level
0,T0,21,177053,402,32694,12,3.87,17807,payment,high
1,T1,24,100729,323,37736,12,12.63,22307,deposit,high
2,T2,25,98355,578,46700,36,11.11,93001,withdrawal,high
3,T3,21,176776,845,47249,48,7.52,77294,deposit,low
4,T4,56,114265,400,4769,48,4.40,41656,payment,high


In [ ]:
def create_prompt(row):

    return f"""
### Financial Risk Classification Task

Customer Age: {row['customer_age']}
Income: {row['income']}
Credit Score: {row['credit_score']}
Loan Amount: {row['loan_amount']}
Loan Term: {row['loan_term']}
Interest Rate: {row['interest_rate']}
Account Balance: {row['account_balance']}
Transaction Type: {row['transaction_type']}

Question: What is the financial risk level?

Answer: {row['risk_level']}
"""

df["text"] = df.apply(create_prompt, axis=1)

df[["text"]].to_json("finance_instruction.json", orient="records", lines=True)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="finance_instruction.json", split="train")

dataset = dataset.train_test_split(test_size=0.2)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer.pad_token = tokenizer.eos_token
model.config.use_cache = False

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj","v_proj"],
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [ ]:
def tokenize(example):

    tokens = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

train_dataset = train_dataset.map(tokenize)
test_dataset = test_dataset.map(tokenize)

Map:   0%|          | 0/3200 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./tinyllama-finance",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=4,
    logging_steps=20,
    fp16=True
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

Step,Training Loss
20,5.630505
40,0.414325
60,0.244433
80,0.225112
100,0.222608
120,0.222308
140,0.220989
160,0.220704
180,0.219670
200,0.219992


TrainOutput(global_step=1600, training_loss=0.2857462403178215, metrics={'train_runtime': 2068.8887, 'train_samples_per_second': 6.187, 'train_steps_per_second': 0.773, 'total_flos': 2.03836329295872e+16, 'train_loss': 0.2857462403178215, 'epoch': 4.0})

In [ ]:
def predict_risk(model, tokenizer, row):

    prompt = f"""
Customer Age: {row['customer_age']}
Income: {row['income']}
Credit Score: {row['credit_score']}
Loan Amount: {row['loan_amount']}
Loan Term: {row['loan_term']}
Interest Rate: {row['interest_rate']}
Account Balance: {row['account_balance']}
Transaction Type: {row['transaction_type']}

Risk Level:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output = model.generate(**inputs, max_new_tokens=10)

    result = tokenizer.decode(output[0], skip_special_tokens=True)

    if "low" in result.lower():
        return "low"
    elif "medium" in result.lower():
        return "medium"
    elif "high" in result.lower():
        return "high"
    else:
        return "medium"

In [ ]:
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

y_true = []
y_pred_before = []

for _,row in df.sample(200).iterrows():

    y_true.append(row["risk_level"])

    pred = predict_risk(base_model, tokenizer, row)

    y_pred_before.append(pred)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
y_pred_after = []

for _,row in df.sample(200).iterrows():

    pred = predict_risk(model, tokenizer, row)

    y_pred_after.append(pred)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

print("===== BEFORE FINE TUNING =====")

print("Accuracy:", accuracy_score(y_true,y_pred_before))
print("Precision:", precision_score(y_true,y_pred_before,average="weighted",zero_division=0))
print("Recall:", recall_score(y_true,y_pred_before,average="weighted",zero_division=0))
print("F1 Score:", f1_score(y_true,y_pred_before,average="weighted",zero_division=0))


print("\n===== AFTER FINE TUNING =====")

print("Accuracy:", accuracy_score(y_true,y_pred_after))
print("Precision:", precision_score(y_true,y_pred_after,average="weighted",zero_division=0))
print("Recall:", recall_score(y_true,y_pred_after,average="weighted",zero_division=0))
print("F1 Score:", f1_score(y_true,y_pred_after,average="weighted",zero_division=0))


print("\nClassification Report:")
print(classification_report(y_true,y_pred_after))

===== BEFORE FINE TUNING =====
Accuracy: 0.08
Precision: 0.03616177070583435
Recall: 0.08
F1 Score: 0.04801416122004357

===== AFTER FINE TUNING =====
Accuracy: 0.27
Precision: 0.46434210526315794
Recall: 0.27
F1 Score: 0.1967427334570192

Classification Report:
              precision    recall  f1-score   support

        high       0.61      0.09      0.15       129
         low       0.09      0.05      0.06        22
      medium       0.25      0.86      0.38        49

    accuracy                           0.27       200
   macro avg       0.32      0.33      0.20       200
weighted avg       0.46      0.27      0.20       200

